In [ ]:
import re, os
import csv
import random
import subprocess
import pathlib as Path
from collections import defaultdict

In [ ]:
software={
    'micapipe_img':'/data_/mica1/01_programs/micapipe-v0.2.0/micapipe_v0.2.3.sif',
    'post_qc_fastsurfer':'/data/mica1/01_programs/MICA-7t/functions/post-qc_fastsurfer.sh',
    
    'q_wrapper': '/host/verges/tank/data/daniel/99_varia/MPN/processing/000_qsubCall.sh',
    '00_PreScrubber': '/host/verges/tank/data/daniel/99_varia/MPN/processing/00_PreScrubber.sh',
}

params = {
    'root':"/host/verges/tank/data/BIDS_MPN",
    'T1wstr': 'UNIT1,inv-1_MP2RAGE,inv-2_MP2RAGE', # comma separate strings for T1w images that will be registered and averaged to give T1_nativepro
    # when sending T1wstr, add 'ses-{ses}_{T1wstr}' to the bids path for each T1w image
    
    'test': False,
    'n_test': 1,
    'verbose': 0,
    
    'queue':"mica.q",
    'threads': 12,
    'qsub_logs': '/host/verges/tank/data/daniel/99_varia/MPN/processing/logs/'
}


In [ ]:
# Wrapper helpers
def get_sub_ses(dir_raw, verbose=0):
    pt_ids = set()
    subject_session_pairs = []
    
    # Read participants.tsv
    pt_tsv = dir_raw / "participants.tsv"
    with open(pt_tsv, newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")

        if "participant_id" not in reader.fieldnames:
            raise ValueError("participants.tsv must contain a 'participant_id' column")

        for row in reader:
            sub = row["participant_id"].strip()
            
            if not sub:
                continue
            pt_ids.add(sub)
            
            sub_dir = dir_raw / sub
            sessions_file = sub_dir / f"{sub}_sessions.tsv"
            
            if sessions_file.exists():
                with open(sessions_file, newline="") as sf:
                    sreader = csv.DictReader(sf, delimiter="\t")

                    if "session_id" not in sreader.fieldnames:
                        raise ValueError(f"{sessions_file} must contain a 'session_id' column")

                    for srow in sreader:
                        session_id = srow["session_id"].strip()
                        if session_id:
                            subject_session_pairs.append((sub, session_id))
            else: # No sessions.tsv
                ses_dirs = sorted(
                    d.name for d in sub_dir.glob("ses-*") if d.is_dir()
                )

                if ses_dirs:
                    for ses in ses_dirs:
                        subject_session_pairs.append((sub, ses))
                else:
                    subject_session_pairs.append((sub, None))
                    pt_ids.add(sub)

    if verbose > 0:
        print(f"{len(subject_session_pairs)} subject-session pairs ({len(pt_ids)} unique participants).")
    return subject_session_pairs

def get_studyName(root):
    datasetName = re.search(r"/BIDS_([^/]+)/", root)
    if datasetName:
        return datasetName.group(1)
    else:
        return None

def get_todo(params):
    sub_ses_pairs = get_sub_ses(Path.Path(params['root']) / "rawdata")
    if params['test']:
        # choose random subset of subject-session pairs for testing
        todo = random.sample(sub_ses_pairs, params['n_test'])
        if params['verbose'] > 0:
            print(f"[PreScrubber] TEST MODE: processing {params['n_test']} subject-session pairs ({todo})")
    else:
        todo = sub_ses_pairs
        if params['verbose'] > 0:
            print(f"[PreScrubber] processing {len(sub_ses_pairs)} subject-session pairs")
    return todo

def get_IDs_imgString(rawdata_dir, pattern, ses='any'):
    """
    Return a list of subject IDs that have at least one file whose
    path matches `pattern` in the specified session.

    rawdata_dir: BIDS rawdata root (contains sub-XXX/ directories)
    pattern: regex string applied to the full file path OR filename
    ses: session label (e.g. 'MPN1Scan1') or 'any' to accept any session
    """
    regex = re.compile(pattern)
    subjects_with_match = set()

    # assuming structure: rawdata_dir/sub-<id>/ses-<ses>/.../<files>
    for sub_name in os.listdir(rawdata_dir):
        if not sub_name.startswith("sub-"):
            continue
        sub_id = sub_name[len("sub-"):]
        sub_path = os.path.join(rawdata_dir, sub_name)

        if not os.path.isdir(sub_path):
            continue

        # collect sessions for this subject
        ses_dirs = [
            d for d in os.listdir(sub_path)
            if d.startswith("ses-") and os.path.isdir(os.path.join(sub_path, d))
        ]

        for ses_dir in ses_dirs:
            ses_label = ses_dir[len("ses-"):]
            if ses != "any" and ses_label != ses:
                continue

            ses_path = os.path.join(sub_path, ses_dir)
            for root, _, files in os.walk(ses_path):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    if regex.search(fpath):
                        subjects_with_match.add(sub_id)
                        # once we found a match for this subject, no need to keep scanning
                        break
                if sub_id in subjects_with_match:
                    break
            if sub_id in subjects_with_match:
                # if ses='any', we can stop scanning other sessions for this subject
                break

    return sorted(subjects_with_match)

def mk_txt(todo, out, sep=','): # each set is line, seperate each item in set with sep
    lines = []
    for sub, ses in todo:
        line = f"{sub}{sep}{ses}"
        lines.append(line)
    with open(out, 'w') as f:
        f.write("\n".join(lines))
    return

def rm_Prefix(sub=None, ses=None):
    if sub is not None and sub.startswith("sub-"):
        sub = sub[4:]
    if ses is not None and ses.startswith("ses-"):
        ses = ses[4:]
    return sub, ses

# Wrappers

def PreScrubber(params, software):
    todo = get_todo(params)
    
    for sub, ses in todo:
        print(f"{sub} @ {ses}")
        
        # prep
        studyName = get_studyName(params['root']) + '_' if get_studyName(params['root']) else None
        jobname = f"{studyName}{sub}_{ses}_00" if studyName is not None else f"{sub}_{ses}_00"
        std_out_path = Path.Path(params['qsub_logs']) / f"{jobname}.o"
        std_err_path = Path.Path(params['qsub_logs']) / f"{jobname}.e"
        T1wStr=params['T1wstr']
        
        cmd = [
            "qsub",
            "-cwd",          # run in current working directory
            "-V",            # export environment
            "-q", params['queue'],     # send job to specific queue
            "-N", jobname,
            "-o", str(std_out_path),
            "-e", str(std_err_path),
            "-terse",
            str(software['q_wrapper']),
            "--threads", str(params['threads']),
            str(software['00_PreScrubber']),
            sub,
            ses,
            params['root'],
            T1wStr,
            software["micapipe_img"]
        ]
        
        print("Submitting:", " ".join(cmd))
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        stdout, stderr = proc.communicate()
        line = stdout.strip()
        print(stdout)
        print(stderr)

        # Extract job ID (assumes standard SGE message format)
        job_id = None
        parts = line.split()
        for p in parts:
            if p.isdigit():
                job_id = p
                break

        if job_id:
            print(f"Submitted job ID: {job_id} for {sub} @ {ses}")
        else:
            print("Warning: could not parse job ID from qsub output")


In [ ]:
# Save all sub-ses pairs to txt file s
todo = get_todo(params)
mk_txt(todo,  "/host/verges/tank/data/daniel/99_varia/MPN/MPN_29jun2026.txt")